In [6]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Senpai's Shelf Orders")      # run once; next time use gc.open("Senpai's Shelf Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "date", "hologram_verified", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1bIRhlax2U5KqRExFeY-5RIujrT7RQxbHcWhfO2yji-E


In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

INVENTORY = {
    "gojo satoru nendoroid": 4500,
    "luffy gear 5 scale figure": 12000,
    "naruto ramen pop": 1500,
    "anya forger chibi": 2200,
    "zoro katana keychain": 450,
}

HOURS = {
    "monday": "10 am to 9 pm",
    "tuesday": "10 am to 9 pm",
    "wednesday": "10 am to 9 pm",
    "thursday": "10 am to 9 pm",
    "friday": "10 am to 9 pm",
    "saturday": "10 am to 10 pm",
    "sunday": "11 am to 7 pm",
}

class Order(BaseModel):
    """A figurine order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal[
        "gojo satoru nendoroid",
        "luffy gear 5 scale figure",
        "naruto ramen pop",
        "anya forger chibi",
        "zoro katana keychain",
    ] = Field(description="Inventory item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Preferred pickup or order date as the customer said it")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one inventory item.

    Args:
        item: Figurine or keychain name, e.g. "gojo satoru nendoroid" or "luffy gear 5 scale figure"
        qty: How many the customer wants
    """
    price = INVENTORY.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not in our catalog. Let me check with the Super Senpai (store owner)!"
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check Senpai's Shelf store hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str) -> str:
    """Save a confirmed figurine order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and preferred date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = INVENTORY[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date, "yes", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."



In [8]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the enthusiastic, slightly nerdy store clerk at "Senpai's Shelf", an anime figurine store.
FACTS:  All figurines are 100% authentic Japanese imports with official hologram stickers. Everything is in rupees.
RULES:  Never promise an exact delivery date. Never accept haggling or give unauthorized discounts.
        Always use get_price for prices and is_open for timings; never guess them.
        If an item or question isn't in FACTS or tools, say "Let me check with the Super Senpai (store owner)!"
        Make sure to ask the customer's name and remember their name and items they want during the chat.
        Before log_order, make sure you have name, item, quantity and preferred date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 50 words, energetic and friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "We're a little busy. Please try again in a minute."

print(chat("Are you open on Sunday? How much are 2 Gojo Satoru Nendoroids?"))
print(chat("I'm Kiran. Please book 2 Gojo Satoru Nendoroids for 24 Dec."))   # check your sheet!



Hey! We’re open Sunday 11 am‑7 pm. Two Gojo Satoru Nendoroids cost Rs 9,000. What’s your name, and would you like to place an order?
Got it, Kiran! Your 2 Gojo Satoru Nendoroids are booked for 24 Dec. Total Rs 9,000. See you soon—happy shopping!


In [9]:
for m in history:
    m.pretty_print()

================================ Human Message =================================

Are you open on Sunday? How much are 2 Gojo Satoru Nendoroids?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_a9b6baa5-9308-4416-b1a8-27d72591c5e2)
 Call ID: fc_a9b6baa5-9308-4416-b1a8-27d72591c5e2
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 11 am to 7 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_e41c9765-e32b-40bc-b89d-5c3f1ca2c6ea)
 Call ID: fc_e41c9765-e32b-40bc-b89d-5c3f1ca2c6ea
  Args:
    item: gojo satoru nendoroid
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x gojo satoru nendoroid = Rs 9000
================================== Ai Message ==================================

Hey! We’re open Sunday 11 am‑7 pm. Two Gojo Sa